In [1]:
import pandas as pd
import numpy as np
import os
from scipy import stats
from sklearn.metrics import roc_auc_score

In [2]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

In [3]:
def ttest_one_var(test_data, feature_col, target_col, equal_var=False):
    """
    Perform an independent t-test for one numeric column against a binary target.
    
    Args:
        test_data (pd.dataframe): Numeric data to test
        feature_col (column name of feature): Numeric feature to test
        target_col (column name of target): Binary target (must have exactly 2 groups)  
        equal_var (bool): If False (default), uses Welch's t-test (unequal variances)
    
    Returns:
        dict: t-statistic, p-value, group means, and effect size (Cohen's d)
    
    Example:
        results = ttest_one_var(df, 'LTV', 'TARGET', equal_var=False)
    """
    # Check for exactly 2 groups
    groups = [0, 1]
    
    # Split data into two groups
    group1 = test_data.loc[test_data[target_col] == groups[0], feature_col].dropna()
    group2 = test_data.loc[test_data[target_col] == groups[1], feature_col].dropna()
    
    # Perform t-test
    t_stat, p_value = stats.ttest_ind(group1, group2, equal_var=equal_var)
    
    # Calculate effect size (Cohen's d)
    n1, n2 = len(group1), len(group2)
    pooled_std = np.sqrt(((n1-1)*group1.std()**2 + (n2-1)*group2.std()**2) / (n1+n2-2))
    cohens_d = (group1.mean() - group2.mean()) / pooled_std
    
    return {
        'feature': feature_col,
        't_statistic': t_stat,
        'p_value': p_value,
        'mean_group1': group1.mean(),
        'mean_group2': group2.mean(),
        'cohens_d': cohens_d,
        'abs_t_stat': abs(t_stat)  # Useful for ranking variables
    }

In [4]:
def calculate_auc_one_var(df, numeric_col, target_col):
    """
    Simplified AUC calculation using sklearn's roc_auc_score.
    Returns json with AUC scores.
    """
    x = df[numeric_col]
    y = df[target_col]
    valid_idx = x.notna() & y.notna()  # Drop missing values

    try:
        auc = roc_auc_score(y[valid_idx], x[valid_idx])
        gini_abs = abs(2 * auc - 1)  # Convert AUC to Gini coefficient
        result = {'variable': numeric_col, 'auc_score': auc, 'gini_abs': gini_abs}
    except ValueError:
        result = {'variable': numeric_col, 'auc_score': None, 'gini_abs': None}
    
    return result

In [5]:
def calculate_numeric_selection_metrics(df, numeric_cols, target_col):
    """
    Calculate AUC and t-test for a numeric feature against a binary target.
    
    Args:
        df (pd.DataFrame): DataFrame containing the feature and target
        numeric_col (str): Name of the numeric feature column
        target_col (str): Name of the binary target column
    
    Returns:
        dict: AUC score and t-test results
    """

    results = []
    for col in numeric_cols:
        auc_result = calculate_auc_one_var(df, col, target_col)
        ttest_result = ttest_one_var(df, col, target_col, equal_var=False)
        missing_perc = df[col].isna().sum()/df.shape[0] * 100
        results.append({
            'variable': col,
            'missing_count': missing_perc,
            'auc_score': auc_result['auc_score'],
            'gini_abs': auc_result['gini_abs'],
            't_statistic': ttest_result['t_statistic'],
            'p_value': ttest_result['p_value'],
            'mean_group1': ttest_result['mean_group1'],
            'mean_group2': ttest_result['mean_group2'],
            'cohens_d': ttest_result['cohens_d']
        })
    
    return pd.DataFrame(results)
    

In [6]:
def logistic_regression_coef(df, feature_cols, target_col):
    """
    Fit a logistic regression model and return coefficients for each feature.
    
    Args:
        df (pd.DataFrame): DataFrame containing features and target
        feature_cols (list): List of feature column names
        target_col (str): Name of the target column
    
    Returns:
        pd.DataFrame: DataFrame with feature names and their coefficients
    """
    X = df[feature_cols]
    y = df[target_col]

    # Handle missing values
    imputer = SimpleImputer(strategy='median')
    X_imputed = imputer.fit_transform(X)

    # Scale features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_imputed)

    # Fit logistic regression model
    model = LogisticRegression(random_state=123, max_iter=10000, penalty='l1', solver='liblinear', C=0.1)
    model.fit(X_scaled, y)

    # Create DataFrame with coefficients
    coef_df = pd.DataFrame({'variable': feature_cols, 'coef': model.coef_[0]})
    
    return coef_df

In [7]:
os.getcwd()
os.chdir('a:\\Dominic\\My Projects\\Home Credit')

In [8]:
base = pd.read_csv('data/Internal/application_train.csv', usecols=['SK_ID_CURR', 'TARGET'])
base = base.astype({'SK_ID_CURR': 'string'})

In [9]:
df = pd.read_parquet('src\\intermediate\\creditcard_processed.parquet')

In [10]:
df_with_target = df#base.merge(df, on='SK_ID_CURR', how='left')

In [11]:
numcols = df_with_target.select_dtypes(include=np.number).columns.to_list()
numcols.remove('TARGET')

In [12]:
numcols

['AMT_BALANCE_3Y_AVG',
 'AMT_CREDIT_LIMIT_ACTUAL_3Y_AVG',
 'AMT_DRAWINGS_ATM_CURRENT_3Y_AVG',
 'AMT_DRAWINGS_CURRENT_3Y_AVG',
 'AMT_DRAWINGS_OTHER_CURRENT_3Y_AVG',
 'AMT_DRAWINGS_POS_CURRENT_3Y_AVG',
 'AMT_INST_MIN_REGULARITY_3Y_AVG',
 'AMT_PAYMENT_CURRENT_3Y_AVG',
 'AMT_PAYMENT_TOTAL_CURRENT_3Y_AVG',
 'AMT_RECEIVABLE_PRINCIPAL_3Y_AVG',
 'AMT_RECIVABLE_3Y_AVG',
 'AMT_TOTAL_RECEIVABLE_3Y_AVG',
 'CNT_DRAWINGS_ATM_CURRENT_3Y_AVG',
 'CNT_DRAWINGS_CURRENT_3Y_AVG',
 'CNT_DRAWINGS_OTHER_CURRENT_3Y_AVG',
 'CNT_DRAWINGS_POS_CURRENT_3Y_AVG',
 'CNT_INSTALMENT_MATURE_CUM_3Y_AVG',
 'UTILIZATION_RATIO_3Y_AVG',
 'PAYMENT_RATIO_3Y_AVG',
 'DRAWING_TO_PYMNT_RATIO_3Y_AVG',
 'INT_COVERAGE_RATIO_3Y_AVG',
 'POS_DRAWING_AMT_RATIO_3Y_AVG',
 'ATM_DRAWING_AMT_RATIO_3Y_AVG',
 'POS_DRAWING_CNT_RATIO_3Y_AVG',
 'ATM_DRAWING_CNT_RATIO_3Y_AVG',
 'ISNULL_DRAWINGS_CURRENT_FLAG_3Y_AVG',
 'AMT_BALANCE_YOY_PCT_CHANGE_3Y_AVG',
 'AMT_CREDIT_LIMIT_ACTUAL_YOY_PCT_CHANGE_3Y_AVG',
 'AMT_DRAWINGS_CURRENT_YOY_PCT_CHANGE_3Y_AVG',


In [13]:
numeric_selection_metrics = calculate_numeric_selection_metrics(df_with_target, numcols, 'TARGET')

In [14]:
coef_df = logistic_regression_coef(df_with_target, numcols, 'TARGET')

In [15]:
numeric_selection_metrics_full = numeric_selection_metrics.merge(coef_df,
                                                                how='left', 
                                                                right_on='variable',
                                                                left_on='variable'
                                                                )

In [16]:
numeric_selection_metrics_full

,variable,missing_count,auc_score,gini_abs,t_statistic,p_value,mean_group1,mean_group2,cohens_d,coef
0,AMT_BALANCE_3Y_AVG,0.0,0.611976,0.223952,-23.415663,1.333285e-117,62514.988075,98476.475779,-0.322155,0.042754
1,AMT_CREDIT_LIMIT_ACTUAL_3Y_AVG,0.0,0.503637,0.007275,1.818129,6.907670e-02,204175.743284,200135.337604,0.020649,-0.069167
2,AMT_DRAWINGS_ATM_CURRENT_3Y_AVG,0.0,0.608426,0.216852,-19.639029,5.142488e-84,7056.117264,11864.889200,-0.281891,0.000000
3,AMT_DRAWINGS_CURRENT_3Y_AVG,0.0,0.607892,0.215783,-17.900373,1.895623e-70,12877.572065,18608.154918,-0.213458,-0.132846
4,AMT_DRAWINGS_OTHER_CURRENT_3Y_AVG,0.0,0.507749,0.015499,-4.032990,5.554790e-05,344.504528,544.760822,-0.055646,0.000000
...,...,...,...,...,...,...,...,...,...,...
367,ACTIVE_PREV_CREDITS_LAST_1Y,0.0,0.505942,0.011883,-6.340961,2.385850e-10,0.970972,0.982878,-0.066784,-0.019357
368,NON_ACTIVE_PREV_CREDITS_LAST_1Y,0.0,0.486595,0.026810,7.113307,1.216096e-12,0.134545,0.107513,0.079134,0.020061
369,TOTAL_PREV_CREDITS,0.0,0.500655,0.001309,-1.211745,2.256426e-01,1.006186,1.007433,-0.015594,0.030501
370,ACTIVE_PREV_CREDITS,0.0,0.500757,0.001513,-1.350393,1.769247e-01,1.005317,1.006769,-0.017247,0.000000


In [17]:
numeric_selection_metrics_full['points'] = ( numeric_selection_metrics_full['coef'] > 0 )* 1 +\
                                           ( numeric_selection_metrics_full['gini_abs'] > 0.1 )* 1 +\
                                           ( numeric_selection_metrics_full['p_value'] < 0.01 )* 1  

In [18]:
numeric_selection_metrics_full.to_excel('src\\Variable-Selection\\creditcard_selection_metrics_full.xlsx', index=False)